# Predicting Student Health Risk

## Data Preprocessing

This notebook prepares the Kaggle dataset for machine learning. It separates the input features and target, removes identifier columns, handles missing values, encodes categorical features, scales numerical features, and creates training and validation datasets.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
current_folder = Path.cwd()

if (current_folder / "data").exists():
    project_folder = current_folder
else:
    project_folder = current_folder.parent

data_folder = project_folder / "data" / "kaggle" / "raw"
models_folder = project_folder / "models"

models_folder.mkdir(exist_ok=True)

print("Project folder")
print(project_folder)

print()

print("Data folder")
print(data_folder)

print()

print("Models folder")
print(models_folder)

Project folder
c:\Users\Admin\Documents\Rakindu's Files\ICBT\Computational Intelligence\Assignment\Student health risk project

Data folder
c:\Users\Admin\Documents\Rakindu's Files\ICBT\Computational Intelligence\Assignment\Student health risk project\data\kaggle\raw

Models folder
c:\Users\Admin\Documents\Rakindu's Files\ICBT\Computational Intelligence\Assignment\Student health risk project\models


In [ ]:
train_file = data_folder / "train.csv"
test_file = data_folder / "test.csv"
submission_file = data_folder / "sample_submission.csv"

train_data = pd.read_csv(train_file)
test_data = pd.read_csv(test_file)
sample_submission = pd.read_csv(submission_file)

print("Training data shape", train_data.shape)
print("Testing data shape", test_data.shape)
print("Submission data shape", sample_submission.shape)

Training data shape (690088, 15)
Testing data shape (295753, 14)
Submission data shape (295753, 2)


In [ ]:
train_copy = train_data.copy()
test_copy = test_data.copy()

print("Dataset copies created successfully")

Dataset copies created successfully


In [ ]:
target_column = "health_condition"

X = train_copy.drop(columns=[target_column])
y = train_copy[target_column]

X_test = test_copy.copy()

print("Input feature shape", X.shape)
print("Target shape", y.shape)
print("Kaggle testing feature shape", X_test.shape)

Input feature shape (690088, 14)
Target shape (690088,)
Kaggle testing feature shape (295753, 14)


In [ ]:
print("Target classes")

for class_name in y.unique():
    print(class_name)

Target classes
unhealthy
at-risk
fit


In [ ]:
training_columns = set(X.columns)
testing_columns = set(X_test.columns)

missing_from_test = training_columns - testing_columns
extra_in_test = testing_columns - training_columns

print("Columns missing from testing data")
print(missing_from_test)

print()

print("Extra columns in testing data")
print(extra_in_test)

Columns missing from testing data
set()

Extra columns in testing data
set()


In [ ]:
identifier_columns = []

for column in X.columns:
    if column.lower() in ["id", "student_id", "record_id"]:
        identifier_columns.append(column)

print("Identifier columns")
print(identifier_columns)

Identifier columns
['id']


In [ ]:
test_identifiers = pd.DataFrame()

if len(identifier_columns) > 0:
    test_identifiers = X_test[identifier_columns].copy()

print("Testing identifiers saved")
print(test_identifiers.head())

Testing identifiers saved
       id
0  690088
1  690089
2  690090
3  690091
4  690092


In [ ]:
if len(identifier_columns) > 0:
    X = X.drop(columns=identifier_columns)
    X_test = X_test.drop(columns=identifier_columns)

print("Training feature shape after removing identifiers", X.shape)
print("Testing feature shape after removing identifiers", X_test.shape)

Training feature shape after removing identifiers (690088, 13)
Testing feature shape after removing identifiers (295753, 13)


In [ ]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Numerical columns")

for column in numerical_columns:
    print(column)

print()

print("Number of numerical columns", len(numerical_columns))

Numerical columns
sleep_duration
heart_rate
bmi
calorie_expenditure
step_count
exercise_duration
water_intake

Number of numerical columns 7


In [ ]:
categorical_columns = X.select_dtypes(
    exclude=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical columns")

for column in categorical_columns:
    print(column)

print()

print("Number of categorical columns", len(categorical_columns))

Categorical columns
diet_type
stress_level
sleep_quality
physical_activity_level
smoking_alcohol
gender

Number of categorical columns 6


In [ ]:
print("Missing values in numerical columns")
print(X[numerical_columns].isnull().sum())

print()

print("Missing values in categorical columns")
print(X[categorical_columns].isnull().sum())

Missing values in numerical columns
sleep_duration         75999
heart_rate              7833
bmi                    13898
calorie_expenditure    52853
step_count             13916
exercise_duration       6901
water_intake           43477
dtype: int64

Missing values in categorical columns
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64


In [ ]:
X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training input shape", X_train.shape)
print("Validation input shape", X_validation.shape)
print("Training target shape", y_train.shape)
print("Validation target shape", y_validation.shape)

Training input shape (552070, 13)
Validation input shape (138018, 13)
Training target shape (552070,)
Validation target shape (138018,)


In [ ]:
print("Original target percentages")
print(y.value_counts(normalize=True).mul(100).round(2))

print()

print("Training target percentages")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print()

print("Validation target percentages")
print(y_validation.value_counts(normalize=True).mul(100).round(2))

Original target percentages
health_condition
at-risk      85.87
unhealthy     8.36
fit           5.77
Name: proportion, dtype: float64

Training target percentages
health_condition
at-risk      85.87
unhealthy     8.36
fit           5.77
Name: proportion, dtype: float64

Validation target percentages
health_condition
at-risk      85.87
unhealthy     8.36
fit           5.77
Name: proportion, dtype: float64


In [ ]:
numerical_preprocessor = Pipeline(
    steps=[
        (
            "missing_value_handler",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("Numerical preprocessing pipeline created")

Numerical preprocessing pipeline created


In [ ]:
categorical_preprocessor = Pipeline(
    steps=[
        (
            "missing_value_handler",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

print("Categorical preprocessing pipeline created")

Categorical preprocessing pipeline created


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_preprocessor,
            numerical_columns
        ),
        (
            "categorical",
            categorical_preprocessor,
            categorical_columns
        )
    ]
)

print("Complete preprocessing pipeline created")

Complete preprocessing pipeline created


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

print("Preprocessing pipeline fitted successfully")
print("Processed training shape", X_train_processed.shape)

Preprocessing pipeline fitted successfully
Processed training shape (552070, 25)


In [ ]:
X_validation_processed = preprocessor.transform(X_validation)

print("Validation data transformed successfully")
print("Processed validation shape", X_validation_processed.shape)

Validation data transformed successfully
Processed validation shape (138018, 25)


In [ ]:
X_test_processed = preprocessor.transform(X_test)

print("Kaggle testing data transformed successfully")
print("Processed testing shape", X_test_processed.shape)

Kaggle testing data transformed successfully
Processed testing shape (295753, 25)


In [ ]:
print("Original training columns", X_train.shape[1])
print("Processed training columns", X_train_processed.shape[1])

print()

print("Original validation columns", X_validation.shape[1])
print("Processed validation columns", X_validation_processed.shape[1])

Original training columns 13
Processed training columns 25

Original validation columns 13
Processed validation columns 25


In [ ]:
preprocessor_file = models_folder / "preprocessing_pipeline.joblib"

joblib.dump(
    preprocessor,
    preprocessor_file
)

print("Preprocessing pipeline saved successfully")
print(preprocessor_file)

Preprocessing pipeline saved successfully
c:\Users\Admin\Documents\Rakindu's Files\ICBT\Computational Intelligence\Assignment\Student health risk project\models\preprocessing_pipeline.joblib


In [ ]:
column_information = {
    "target_column": target_column,
    "identifier_columns": identifier_columns,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "original_feature_columns": X.columns.tolist()
}

column_file = models_folder / "column_information.joblib"

joblib.dump(
    column_information,
    column_file
)

print("Column information saved successfully")
print(column_file)

Column information saved successfully
c:\Users\Admin\Documents\Rakindu's Files\ICBT\Computational Intelligence\Assignment\Student health risk project\models\column_information.joblib


In [ ]:
print("Preprocessing pipeline exists")
print(preprocessor_file.exists())

print()

print("Column information exists")
print(column_file.exists())

Preprocessing pipeline exists
True

Column information exists
True


## Preprocessing summary

The target variable was separated from the input features.

The `id` column was removed from model training.

The dataset contains 7 numerical input features and 6 categorical input features.

Numerical missing values were filled using the median.

Categorical missing values were filled using the most frequent value.

Numerical features were standardised using `StandardScaler`.

Categorical features were converted using One-Hot Encoding.

The data was divided into 80 percent training data and 20 percent validation data using stratification.

The preprocessing pipeline was fitted only on the training data to prevent data leakage.

The processed training dataset contains 552,070 rows and 25 columns.

The processed validation dataset contains 138,018 rows and 25 columns.

The processed Kaggle testing dataset contains 295,753 rows and 25 columns.

The preprocessing pipeline and column information were saved successfully.